In [1]:
import pandas as pd
import pyomo.environ as pyo
from food_security import optimizer

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / 'OneDrive - Stichting Deltares/Documents - International Delta Toolset/Food security'

In [3]:
input_file = src_dir / 'input_all.xlsx'

crop_df = pd.read_excel(input_file, sheet_name='crop_id')
trade_df = pd.read_excel(input_file, sheet_name='trade')
production_df = pd.read_excel(input_file, sheet_name='production_d')
production_df = production_df[production_df['year'] == 2020]
price_df = pd.read_excel(input_file, sheet_name='market_price')
price_df = price_df.fillna(180)
theta_df = pd.read_excel(input_file, sheet_name='plant_based_diet_goal')
energy_df = pd.read_excel(input_file, sheet_name='calories')
income_df = pd.read_excel(input_file, sheet_name='income')
price_goal_df = pd.read_excel(input_file, sheet_name='price_goal')

population = 106540000

In [4]:
income_df = income_df.merge(price_goal_df, on='income_quintiles')
income_df['p_spent_food_max'] = income_df['p_spent_food_max'] / 100

price_df = price_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["usd_per_tonnes"].mean()

production_df = production_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["yield"].sum()

trade_df = trade_df[trade_df['crop_name_fao'].isin(production_df['crop_name_fao'])]

energy_df = energy_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["calories_kcal_kg"].mean()

input_df = (
    production_df
    .merge(price_df, on="crop_name_fao", how="left")
    .merge(trade_df, on="crop_name_fao", how="left")
    .merge(energy_df, on="crop_name_fao", how="left")
)

# Convert trade from tonnes to kg
input_df["import"] = input_df["import"] * 1000
input_df["export"] = input_df["export"] * 1000

input_df["quantity"] = input_df["yield"] + input_df["import"] - input_df["export"]

class_map = (
    crop_df
    .drop_duplicates(subset=["crop_name_fao"])
    .set_index("crop_name_fao")["class"]
)

input_df["class"] = input_df["crop_name_fao"].map(class_map)
input_df = input_df[input_df['class'] != '-']
input_df["quantity"] = input_df['quantity'].clip(lower=0, upper=None)
# Convert to daily availability
input_df["quantity"] = input_df["quantity"] / 365.25
input_df["usd_per_kg"] = input_df["usd_per_tonnes"] / 1000

In [5]:
FSOptimizer = optimizer.FSOptimizer(input_df=input_df, income_df=income_df, population=population, MDER=1806.61, lambda_g={
            g: 1.0 for g in ["carb", "protein", "fat", "fruitvege"]
        })
results = FSOptimizer.solve()

In [6]:
results[0]

,5,4,3,2,1
Barley,2.801571e+05,6.223864e+04,9.345774e+04,7.750228e+04,3.598519e+03
"Beans, dry",0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Cabbages,4.823396e+00,3.928162e+06,3.462118e+06,4.492528e+04,0.000000e+00
Chick-peas,1.462113e+04,8.254257e-03,2.707248e-03,1.250322e-03,5.929146e-04
Cucumbers and gherkins,1.205767e+00,1.607741e+00,6.560062e+02,1.375934e+05,1.784192e+01
Eggplants (aubergines),5.869707e+06,6.725826e+06,4.613563e+00,1.136224e-01,4.874498e-02
"Groundnuts, Shelled",2.885960e+05,5.125492e-03,1.685655e-03,7.776562e-04,3.683981e-04
Lemons and limes,1.607732e+00,2.411623e+00,6.111716e+03,1.020873e+06,8.537924e+06
Linseed,9.597946e+04,5.913491e-03,1.942657e-03,8.967714e-04,4.250624e-04
Maize (incl. 0067 and 0068),9.213795e+05,5.303846e+06,5.382405e+06,9.240512e+05,1.560059e+03


In [7]:
results[1]

,5,4,3,2,1
Barley,0.013148,0.002921,0.004386,0.003637,0.000169
"Beans, dry",0.000000,0.000000,0.000000,0.000000,0.000000
Cabbages,0.000000,0.184351,0.162480,0.002108,0.000000
Chick-peas,0.000686,0.000000,0.000000,0.000000,0.000000
Cucumbers and gherkins,0.000000,0.000000,0.000000,0.006457,0.000000
Eggplants (aubergines),0.275470,0.315648,0.000000,0.000000,0.000000
"Groundnuts, Shelled",0.013544,0.000000,0.000000,0.000000,0.000000
Lemons and limes,0.000000,0.000000,0.000287,0.047910,0.400691
Linseed,0.004504,0.000000,0.000000,0.000000,0.000000
Maize (incl. 0067 and 0068),0.043241,0.248913,0.252600,0.043366,0.000000


In [8]:
results[2]

,5,4,3,2,1
objective function,0.467347,0.633921,0.643441,0.648005,0.647386
spending_pp,0.563591,0.447934,0.440029,0.379537,0.431840
percentage_spending,0.046966,0.049770,0.055004,0.063256,0.107960
consumed_energy_pp,1806.610040,1806.610159,1807.047821,1806.852961,1806.610211
carb_fraction,0.857749,0.943921,0.953442,0.958005,0.957386
fat_fraction,0.071085,0.000000,0.000000,0.000000,0.000000
fruitvege_fraction,0.061848,0.056079,0.046558,0.041995,0.042614
protein_fraction,0.009318,0.000000,0.000000,0.000000,0.000000
carb_kg,0.732543,0.587281,0.590933,0.688286,0.713876
fat_kg,0.024016,0.000000,0.000000,0.000000,0.000000
